# 05. Production Serving Dynamic Batcher PD Disaggregation and Streaming

- **Category:** 17_deployment_low_level_and_capstones
- **Lineage & Citations:** Crankshaw et al. (2017, *Clipper*), Patel et al. (2024, *Splitwise*), Zhong et al. (2024, *DistServe*), Agrawal et al. (2024, *Sarathi-Serve*).
- **Purpose:** Implement and verify production LLM serving architectures with dynamic micro-batching, prefill-decode (PD) disaggregation, KV-cache handoff, and token streaming.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Text Generation Decoding and Structured Grammar Sampling](04_text_generation_decoding_and_structured_grammar_sampling.ipynb) $\to$ `🔗 Downstream Syntheses:` [Torch Dispatch Modes FLOP Counter and Memory Tracker](06_torch_dispatch_modes_flop_counter_and_memory_tracker.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not colocate massive unchunked prefill requests into an active decode batch without chunked prefill or PD disaggregation. Head-of-line blocking spikes inter-token generation latency (TPOT) by over $20\times$, violating real-time streaming SLAs.

## Mathematical Foundations & Governing Equations

$$
Y_{\text{prefill}} = \text{softmax}\left( \frac{Q_{\text{prefill}} K_{\text{prefill}}^T}{\sqrt{D}} + M_{\text{causal}} \right) V_{\text{prefill}} W_o + b_o
$$
$$
y_{\text{decode}} = \text{softmax}\left( \frac{q_{\text{new}} [K_{\text{cache}}; k_{\text{new}}]^T}{\sqrt{D}} \right) [V_{\text{cache}}; v_{\text{new}}] W_o + b_o \equiv Y_{\text{full}}[:, -1:, :]
$$

### Asymptotic Complexity & Disaggregation Advantages
- **Prefill Complexity:** Compute-bound $O(S^2 D)$ operations optimized for high-utilization matrix engines.
- **Decode Complexity:** Memory-bandwidth-bound $O(S D)$ operations requiring high HBM streaming rate.

In [ ]:
import math
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/17_deployment_low_level_and_capstones/05_production_serving_dynamic_batcher_pd_disaggregation_and_streaming.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def prefill_step(x, wq, wk, wv, wo, bo):
    b, s, d = x.shape
    q, k, v = x @ wq, x @ wk, x @ wv
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(d)
    mask = torch.triu(torch.full((s, s), -10000.0, device=x.device), diagonal=1)
    attn = F.softmax(scores + mask, dim=-1)
    return attn @ v @ wo + bo, k, v

def decode_step(x_new, k_cache, v_cache, wq, wk, wv, wo, bo):
    b, _, d = x_new.shape
    q_new, k_new, v_new = x_new @ wq, x_new @ wk, x_new @ wv
    new_k = torch.cat([k_cache, k_new], dim=1)
    new_v = torch.cat([v_cache, v_new], dim=1)
    scores = (q_new @ new_k.transpose(-2, -1)) / math.sqrt(d)
    attn = F.softmax(scores, dim=-1)
    return attn @ new_v @ wo + bo, new_k, new_v

def schedule_dynamic_batch(seq_lengths, max_tokens=16):
    batches, current_batch, current_tokens = [], [], 0
    for idx, length in enumerate(seq_lengths):
        if current_tokens + length > max_tokens and current_batch:
            batches.append(current_batch)
            current_batch, current_tokens = [idx], length
        else:
            current_batch.append(idx)
            current_tokens += length
    if current_batch: batches.append(current_batch)
    return batches

In [ ]:
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

y_prefill, k_cache, v_cache = prefill_step(weights["x_prefill"], weights["wq"], weights["wk"], weights["wv"], weights["wo"], weights["bo"])
assert float(torch.max(torch.abs(y_prefill - weights["y_prefill"]))) < 1e-5
assert float(torch.max(torch.abs(k_cache - weights["k_cache"]))) < 1e-5
assert float(torch.max(torch.abs(v_cache - weights["v_cache"]))) < 1e-5

y_decode, new_k, new_v = decode_step(weights["x_decode"], k_cache, v_cache, weights["wq"], weights["wk"], weights["wv"], weights["wo"], weights["bo"])
assert float(torch.max(torch.abs(y_decode - weights["y_decode"]))) < 1e-5
assert float(torch.max(torch.abs(new_k - weights["new_k"]))) < 1e-5
assert float(torch.max(torch.abs(new_v - weights["new_v"]))) < 1e-5

# Mathematical equivalence proof: incremental decode equals last slice of full causal forward
assert float(torch.max(torch.abs(y_decode - weights["y_full"][:, -1:, :]))) < 1e-5
print("All PD Disaggregation & Serving parity checks passed (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

benchmarks = [
    benchmark_fn(lambda: linear_anchor(weights["test_input"], weights["weight"], weights["bias"]), iters=50, name="Linear Anchor"),
    benchmark_fn(lambda: prefill_step(weights["x_prefill"], weights["wq"], weights["wk"], weights["wv"], weights["wo"], weights["bo"]), iters=50, name="Prefill Step"),
    benchmark_fn(lambda: decode_step(weights["x_decode"], weights["k_cache"], weights["v_cache"], weights["wq"], weights["wk"], weights["wv"], weights["wo"], weights["bo"]), iters=50, name="Decode Step"),
]
print(format_benchmark_table(benchmarks))

# SVG Visualization of PD Disaggregation
svg = """<svg width="480" height="80" xmlns="http://www.w3.org/2000/svg">
  <rect width="480" height="80" fill="#1e1e2e" rx="8"/>
  <rect x="10" y="20" width="120" height="40" fill="#313244" rx="4" stroke="#89b4fa" stroke-width="1.5"/>
  <text x="70" y="44" fill="#cdd6f4" font-size="11" text-anchor="middle">Prefill Worker</text>
  <path d="M135 40 L195 40" stroke="#f9e2af" stroke-width="2" stroke-dasharray="4,2"/>
  <text x="165" y="32" fill="#f9e2af" font-size="9" text-anchor="middle">KV DMA</text>
  <rect x="200" y="20" width="120" height="40" fill="#313244" rx="4" stroke="#a6e3a1" stroke-width="1.5"/>
  <text x="260" y="44" fill="#cdd6f4" font-size="11" text-anchor="middle">Decode Worker</text>
  <path d="M325 40 L365 40" stroke="#a6e3a1" stroke-width="2"/>
  <rect x="370" y="20" width="100" height="40" fill="#313244" rx="4" stroke="#cba6f7" stroke-width="1.5"/>
  <text x="420" y="44" fill="#cdd6f4" font-size="11" text-anchor="middle">Token Stream</text>
</svg>"""
from IPython.display import display, HTML
display(HTML(svg))

In [ ]:
# Pathological Misuse: Head-of-Line Blocking without Disaggregation vs Independent Micro-Batches
incoming_lengths = [4, 4, 32, 2, 2]  # Request #2 is a 32-token prefill prompt

# Naive colocated batching creates a massive token burst exceeding budget
naive_batch_tokens = sum(incoming_lengths)  # 44 tokens

# Scheduled dynamic micro-batcher isolates the 32-token prefill into its own chunked pass
scheduled = schedule_dynamic_batch(incoming_lengths, max_tokens=16)

print(f"Naive Single Batch Token Burst: {naive_batch_tokens} tokens (Causes decode starvation!)")
print(f"Dynamic Micro-Batches:         {len(scheduled)} batches -> {scheduled}")
assert len(scheduled) > 1 and max(len(b) for b in scheduled) < len(incoming_lengths), "Pathology demonstrated!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Prefill-Decode Decoupling:** Separating compute-heavy prefill nodes from memory-bound decode nodes eliminates head-of-line blocking stalls.
2. **Lossless Equivalence:** Incrementally appending tokens to a transferred KV-cache reproduces the exact output of full causal forward passes.
3. **Token Budget Micro-Batching:** Dynamic scheduling bounds peak activations per iteration to prevent out-of-memory crashes.

---
### Cross-Domain Generalisation Matrix
| Domain | PD Disaggregation Analogue | Architectural Objective |
| :--- | :--- | :--- |
| **Video Streaming Transcoding** | Keyframe I-frame encoding vs P/B-frame delta streaming | Isolate heavy initial frame compression from lightweight motion deltas |
| **Distributed Database Engines** | Batch columnar index scanning vs low-latency row point lookups | Preserve sub-millisecond query response while running analytics |
| **Financial High-Frequency Trading** | Heavy overnight risk modeling vs real-time sub-microsecond order execution | Decouple compute-bound portfolio calibration from order book streaming |

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [05_production_serving_dynamic_batcher_pd_disaggregation_and_streaming.md](../../markdown/17_deployment_low_level_and_capstones/05_production_serving_dynamic_batcher_pd_disaggregation_and_streaming.md)
- **PyTorch 2.x Module:** [05_production_serving_dynamic_batcher_pd_disaggregation_and_streaming.py](../../pytorch2/17_deployment_low_level_and_capstones/05_production_serving_dynamic_batcher_pd_disaggregation_and_streaming.py)
- **Functional JAX Module:** [05_production_serving_dynamic_batcher_pd_disaggregation_and_streaming.py](../../jax/17_deployment_low_level_and_capstones/05_production_serving_dynamic_batcher_pd_disaggregation_and_streaming.py)
- **Native Rust Candle Track:** [05_production_serving_dynamic_batcher_pd_disaggregation_and_streaming.rs](../../candle/17_deployment_low_level_and_capstones/05_production_serving_dynamic_batcher_pd_disaggregation_and_streaming.rs)
- **Hardware Kernel (CPU / WGSL):** [05_production_serving_dynamic_batcher_pd_disaggregation_and_streaming.rs](../../kernels/17_deployment_low_level_and_capstones/05_production_serving_dynamic_batcher_pd_disaggregation_and_streaming.rs)
- **Golden SafeTensors:** [05_production_serving_dynamic_batcher_pd_disaggregation_and_streaming.safetensors](../../weights/17_deployment_low_level_and_capstones/05_production_serving_dynamic_batcher_pd_disaggregation_and_streaming.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 04. Text Generation Decoding and Structured Grammar Sampling](04_text_generation_decoding_and_structured_grammar_sampling.ipynb) | [Master Index](../../README.md#17_deployment_low_level_and_capstones) | [Next: 06. Torch Dispatch Modes FLOP Counter and Memory Tracker](06_torch_dispatch_modes_flop_counter_and_memory_tracker.ipynb) &rarr;